# Fase 4 · Visualización y comunicación de resultados
## Licitaciones públicas de ChileCompra · MCDI500 · Grupo 2

Este cuaderno presenta y comunica los resultados descriptivos del proyecto a partir de las salidas consolidadas y validadas de las fases anteriores (`data/processed/`), respondiendo las preguntas definidas en `F1/F1_Definicion.ipynb` (sección 3):

1. **Coincidencia y brecha por moneda:** en qué proporción de las licitaciones comparables coincide el monto estimado registrado con el total de las ofertas ganadoras y, cuando difieren, cuál es el sentido y la magnitud de la brecha en cada moneda (`CLP`, `CLF`, `USD`, `UTM`).
2. **Comparación según tamaño del proveedor ganador:** cómo cambian la coincidencia y la brecha según `TamanoProveedorOrdinal` (`Micro < Pequeña < Mediana < Grande`, manteniendo `NoClasificado` fuera de la escala).

**Alcance y arquitectura.** Siguiendo `docs/arquitectura.md`, toda la lógica de cálculo e implementación gráfica reside en **`src/comunicacion.py`**: este cuaderno únicamente importa y llama a sus funciones sobre los cuatro archivos de `data/processed/` mediante lectura selectiva (`sep=";"` y `usecols`), calculando tasas con intervalos de Wilson al `95 %`, razón (`Ganadoras / Estimado`), log-razón (`ln(razón)`), sentido de brecha, medianas e IQR, tabla comparativa por tamaño y exportación de figuras (`guardar_figura`). El enfoque es descriptivo sobre el corte de enero de 2025, conservando las monedas separadas sin conversiones externas.

**Cómo ejecutar.** Genera primero las referencias en `data/processed/` (ejecutando `F2` y `F3`) y luego ejecuta las celdas de este cuaderno con el kernel **Python (grupo2-mcdi500)**.


## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías utilizadas en esta fase (`pandas`, `numpy` y `matplotlib`) coinciden con las versiones declaradas en `requirements.txt`.


In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 4, con el nombre que tienen en requirements.txt.
PAQUETES_F4 = ["pandas", "numpy", "matplotlib"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas."""
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({
            "Paquete": paquete,
            "Declarada": declaradas.get(paquete),
            "Instalada": instalada,
            "Coincide": instalada == declaradas.get(paquete),
        })
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F4)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas en requirements.txt.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")


Intérprete: .venv\Scripts\python.exe | Python 3.15.2 | Windows
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True
2,matplotlib,3.11.2,3.11.2,True


[OK] Las 3 versiones coinciden con requirements.txt


## 1. Configurar el proyecto

Localiza la raíz del repositorio, agrega `src/` al camino de importación, importa el módulo compartido `src/comunicacion.py` y verifica que existan los cuatro archivos versionados en `data/processed/` (`04LicitacionesEne_procesado.csv`, `04LicitacionesEne_brecha_por_licitacion.csv`, `diccionario_f3.csv` y `parametros_f3.csv`), junto con la ruta de salida reservada para las figuras de la fase (`F4/figuras/`).


In [2]:
# Configurar rutas, importar src/comunicacion.py y verificar los cuatro archivos procesados.
inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / "src/comunicacion.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("No se encontró src/comunicacion.py: abre el notebook dentro del repositorio.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.comunicacion import (
    ARCHIVOS_PROCESADOS,
    COLUMNAS_LICITACIONES_F4,
    COLUMNAS_OFERTAS_F4,
    asignar_tamano_ganador,
    cargar_artefactos,
    graficar_coincidencia_y_sentido,
    graficar_distribucion_brecha,
    graficar_resultados_por_tamano,
    guardar_figura,
    intervalo_wilson,
    preparar_comparables,
    resumen_magnitud_por_moneda,
    resumen_sentido_por_moneda,
    tabla_resumen_por_tamano,
    tasa_por_grupo,
)

carpeta_procesados = raiz / "data/processed"
carpeta_figuras = raiz / "F4/figuras"

for clave, nombre_archivo in ARCHIVOS_PROCESADOS.items():
    ruta_archivo = carpeta_procesados / nombre_archivo
    if not ruta_archivo.is_file():
        raise FileNotFoundError(f"Falta {ruta_archivo.relative_to(raiz)}. Ejecuta F2 y F3 primero.")
    print(f"[OK] {clave:<12} -> {ruta_archivo.relative_to(raiz)}")

print(f"Columnas configuradas con usecols (ofertas): {len(COLUMNAS_OFERTAS_F4)}")
print(f"Columnas configuradas con usecols (licitaciones): {len(COLUMNAS_LICITACIONES_F4)}")
print(f"Directorio previsto para figuras: {carpeta_figuras.relative_to(raiz)}")


[OK] ofertas      -> data\processed\04LicitacionesEne_procesado.csv
[OK] licitaciones -> data\processed\04LicitacionesEne_brecha_por_licitacion.csv
[OK] diccionario  -> data\processed\diccionario_f3.csv
[OK] parametros   -> data\processed\parametros_f3.csv
Columnas configuradas con usecols (ofertas): 7
Columnas configuradas con usecols (licitaciones): 11
Directorio previsto para figuras: F4\figuras
